In [ ]:
import tarfile
import urllib

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
from mlxtend.plotting import plot_decision_regions

import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, confusion_matrix, r2_score, mean_squared_error, classification_report, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Configuración matplotlib
# ==============================================================================
plt.rcParams['image.cmap'] = "bwr"
#plt.rcParams['figure.dpi'] = "100"
plt.rcParams['savefig.bbox'] = "tight"
style.use('ggplot') or plt.style.use('ggplot')

# Configuración warnings
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

<h3>Extracción del repositorio</h3>
El siguiente código descarga y lee el dataset directamente del archivo comprimido .tar.gz<br/>
Otra opción es descargar el repositorio manualmente, descomprimirlo y usar pd.<span style="color:green">read_csv</span>()

In [ ]:
uci_tcga_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00401/"
)
archive_name = "TCGA-PANCAN-HiSeq-801x20531.tar.gz"

# build the url
full_download_url = urllib.parse.urljoin(uci_tcga_url, archive_name)

# download the file
r = urllib.request.urlretrieve(full_download_url, archive_name)

# extract the data from the archive
tar = tarfile.open(archive_name, "r:gz")
tar.extractall()
tar.close()

In [ ]:
datafile = "TCGA-PANCAN-HiSeq-801x20531/data.csv"
labels_file = "TCGA-PANCAN-HiSeq-801x20531/labels.csv"
data = np.genfromtxt(
    datafile, delimiter=",", usecols=range(1, 20532), skip_header=1
)
true_label_names = np.genfromtxt(
    labels_file, delimiter=",", usecols=(1,), skip_header=1, dtype=str
)

In [ ]:
data[:5, :3]

In [ ]:
data.shape

In [ ]:
true_label_names[:5]

<p>Las variables del dataset contiene todos los valores de expresión génica de 20.531 genes. Los true_label_names son los tipos de cáncer para cada una de las 881 muestras. El primer registro en data corresponde con la primera etiqueta en true_labels.</p>
<p>Las etiquetas son cadenas que contienen abreviaturas de tipos de cáncer:
<ul>
    <li><b>BRCA</b>: Carcinoma invasivo de mama (breast)</li>
    <li><b>COAD</b>: adenocarcinoma de colon</li>
    <li><b>KIRC</b>: Carcinoma renal de células claras de riñón (kidney)</li>
    <li><b>LUAD</b>: adenocarcinoma de pulmón (lung)</li>
    <li><b>PRAD</b>: adenocarcinoma de próstata (prostate)</li>
</ul>
Para usar estas etiquetas en los métodos de evaluación, primero debe convertir las abreviaturas a números enteros con LabelEncoder:

In [ ]:
#Codificación de categorias a números
label_encoder = LabelEncoder()
true_labels = label_encoder.fit_transform(true_label_names)
true_labels[:5]

Como label_encoder se ha ajustado a los datos, puede ver las clases únicas representadas mediante <span style="color:green">.classes_</span>. Almacene la longitud de la matriz en la variable <span style="color:green">n_clusters</span> para su uso posterior:

In [ ]:
label_encoder.classes_

<H3>Escalar los datos</h3>

In [ ]:
scaler_minmax=MinMaxScaler()
X= scaler_minmax.fit_transform(data)
X[:5, :3]

<h4>Analisis de componentes principales PCA</h4>

<p>Las técnicas de <span style="color:red">reducción de la dimensionalidad</span> ayudan a abordar un problema con los algoritmos de aprendizaje automático conocido como la maldición de la dimensionalidad. En resumen, a medida que aumenta el número de características, el espacio de características se vuelve escaso. Esta escasez dificulta que los algoritmos encuentren objetos de datos cerca uno del otro en un espacio de mayor dimensión. Dado que el conjunto de datos de expresión génica tiene más de 20 000 características, califica como un gran candidato para la reducción de la dimensionalidad.</p>
<p>El <span style="color:red">análisis de componentes principales (PCA)</span> transforma los datos de entrada proyectándolos en un número menor de dimensiones llamadas componentes. Los componentes capturan la variabilidad de los datos de entrada a través de una combinación lineal de las características de los datos de entrada.</p>

In [ ]:
pca=PCA(n_components=7) # del ejercicio de PCA
pca.fit(X) # obtener los componentes principales
X_pca=pca.transform(X) # convertimos nuestros datos con las nuevas dimensiones de PCA
 
print("shape of X_pca", X_pca.shape)
expl = pca.explained_variance_ratio_
print(expl)
print('suma:',sum(expl[0:6]))

<H3>Separamos el dataset en entrenamiento y pruebas</H3>

In [ ]:
X_pca.shape[0]*0.8

In [ ]:
Xtrain=X_pca[:641,:7]
Xtrain[:5,:7]

In [ ]:
Xtest=X_pca[642:800,:7]
Ytrain=true_labels[:641]
Ytest=true_labels[642:800]

In [ ]:
#si en lugar de array multiple fuera dataframe 
#X = df_breast_cancer.drop('target',axis=1)
#Y = df_breast_cancer['target']
#Xtrain, Xtest, Ytrain, Ytest = train_test_split(X,Y.values.reshape(-1,1),test_size=0.2, random_state=42)

<h3>Creamos el modelo</h3>

In [ ]:
# Creación del modelo SVM lineal
# ==============================================================================
modelo = SVC(C = 100, kernel = 'linear', random_state=123)
modelo.fit(Xtrain, Ytrain)

<h4>Probamos el modelo</h4>

In [ ]:
# Predicciones test
# ==============================================================================
predicciones = modelo.predict(Xtest)
predicciones

In [ ]:
# Accuracy de test del modelo 
# ==============================================================================
accuracy = accuracy_score(
            y_true    = Ytest,
            y_pred    = predicciones,
            normalize = True
           )
print("")
print(f"El accuracy de test es: {100*accuracy}%")

In [ ]:
#Predecimos el test y utilizamos métricas de clasificacion para comprobar nuestros modelos en Train
preds_test = modelo.predict(Xtest)
print('CLASSIFICATION IN TRAIN')
print("----------------------------------------")
print('CONFUSION MATRIX:\n', confusion_matrix(preds_test, Ytest))
print("----------------------------------------")
#print('F1 SCORE:\n', f1_score(preds_test, Ytest))
print("----------------------------------------")
print('CLASSIFICATION REPORT\n',classification_report(preds_test, Ytest))

In [ ]:
# Clasificación con el modelo SVM
# ==============================================================================
true_labels_test=true_labels[642:800]

# Representación gráfica: grupos originales vs clusters creados
# ==============================================================================
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

# Grupos originales
for i in np.unique(true_labels):
    ax[0].scatter(
        x = Xtest[true_labels_test == i, 0],
        y = Xtest[true_labels_test == i, 1], 
        c = plt.rcParams['axes.prop_cycle'].by_key()['color'][i],
        marker    = 'o',
        edgecolor = 'black', 
        label= label_encoder.classes_[i]  
    )
ax[0].set_title('Etiquetas reales')
ax[0].legend();

for i in np.unique(predicciones):
    ax[1].scatter(
        x = Xtest[predicciones == i, 0],
        y = Xtest[predicciones == i, 1], 
        c = plt.rcParams['axes.prop_cycle'].by_key()['color'][i],
        marker    = 'o',
        edgecolor = 'black', 
        label= f"Grupo {label_encoder.classes_[i]}"
    )
    
# Vectores soporte
#ax[1].scatter(
#    modelo.support_vectors_[:, 0],
#    modelo.support_vectors_[:, 1],
#    s=20, linewidth=1,
#    facecolors='none', edgecolors='black'
#)
    
ax[1].set_title('Clasificación generada con KNN')
ax[1].legend();

En este ejemplo el kernel lineal nos dio una precisión del 100% por lo que ya no tiene caso probar con algún otro kernel de <b>SVM</b>.

In [ ]:
modelo.support_vectors_

In [ ]:
modelo.support_vectors_.shape